# Hybrid Rocket Trajectory Simulator

A point-mass, 2-D trajectory model for sounding rockets with hybrid propulsion.  
Given a **target apogee**, the simulator finds all viable (thrust, burn time) pairs,  
then runs a full high-fidelity trajectory for any selected design point.

---

### Key equations

| | Expression |
|---|---|
| Throat area | $A_t = T_{sl} / (P_c \cdot C_F)$ |
| Exit area   | $A_u = A_t \cdot ER$ |
| Thrust (altitude-corrected) | $T(h) = \dot{m}\,c_0 + (P_{sl} - P_{amb})\,A_u$ |
| Propellant mass flow | $\dot{m} = T_{sl}/c_0 = M_p/t_b$ |
| Structural scaling | $M_s = \varepsilon\,M_p\,/\,(1-\varepsilon)$ |
| Mass ratio | $C_2 = M_p/M_s = (1-\varepsilon)/\varepsilon$ |
| Gravity turn | $\dot{\alpha} = -\cos(\alpha)\,g\,/\,V$ |
| Integrator | Heun predictor-corrector, $\Delta t = 0.01$ s |

### Assumptions and limitations
- Nozzle exit area scales with thrust: $A_u \propto T_{sl}$ (constant $P_c$, $C_F$, $ER$)
- Constant thrust profile (flat $\dot{m}$) and constant $c_0$ throughout burn
- ISA atmosphere valid to ~47 km; results above that altitude are approximate
- Natural cubic spline $C_d$(Mach) from a 39-point table (geometry-specific)
- No winds, no Earth rotation, no active guidance
- Structural fraction $\varepsilon$ is constant (linear scaling within a vehicle family)

---


## User Inputs - edit only this cell

In [ ]:
# Vehicle geometry
epsilon  = 0.54     # structural fraction  Ms/M0  (typical range: 0.20 - 0.55)
d_cm     = 20.0     # airframe diameter  [cm]
alfa0    = 85.0     # launch angle from horizontal  [deg]

# Propulsion
c0  = 2138.58   # effective exhaust velocity  [m/s]  (= Isp x g)
Pc  = 30e5      # chamber pressure  [Pa]
CF  = 1.50      # thrust coefficient  (typical N2O/HTPB: 1.4 - 1.6)
ER  = 4.0       # nozzle expansion ratio  A_exit / A_throat

# Mission target
H_target_km = 20.0   # apogee target  [km]

# Feasibility curve - thrust range to explore
Tsl_min_kN = 2.0
Tsl_max_kN = 4.0
n_points   = 10


## Physics models - do not edit

In [ ]:
import numpy as np
import math
import matplotlib.pyplot as plt
from scipy.interpolate import CubicSpline
from scipy.optimize import brentq

PI    = math.pi
ATM   = 101300.0
g     = 9.81
R     = 287.0
GAMMA = 1.4
rad   = PI / 180.0

_T0 = 288.15; _L1 = -0.0065
_T1 = 216.65; _L3 =  0.001;  _T3 = _T1 + _L3 * 12000
_L4 =  0.0028; _T4 = _T3 + _L4 * 15000
_k  = g / R


def air_temperature(z):
    """Return ISA temperature at altitude z [m]."""
    if   z <= 11000: return _T0 + _L1 * z
    elif z <= 20000: return _T1
    elif z <= 32000: return _T1 + _L3 * (z - 20000)
    elif z <= 47000: return _T3 + _L4 * (z - 32000)
    else:            return _T4


def air_pressure(z):
    """Return ISA static pressure at altitude z [m], in Pa."""
    T = air_temperature
    if z <= 11000:
        p = (T(z) / _T0) ** (-_k / _L1)
    elif z <= 20000:
        A = (1 / _L1) * math.log(_T1 / _T0)
        p = math.exp(-_k * (A + (z - 11000) / _T1))
    elif z <= 32000:
        A = (1 / _L1) * math.log(_T1 / _T0) + 9000 / _T1
        p = math.exp(-_k * (A + math.log(T(z) / _T1) / _L3))
    elif z <= 47000:
        A = (1 / _L1) * math.log(_T1 / _T0) + 9000 / _T1 + (1 / _L3) * math.log(_T3 / _T1)
        p = math.exp(-_k * (A + math.log(T(z) / _T3) / _L4))
    else:
        p = 1e-10
    return p * ATM


def air_density(z):
    """Return ISA air density at altitude z [m], in kg/m3."""
    return air_pressure(z) / air_temperature(z) / R


def nozzle_exit_area(Tsl, Pc, CF, ER):
    """
    Compute nozzle exit area that scales with thrust.

    The throat area is derived from thrust, chamber pressure and thrust
    coefficient.  The exit area follows from the expansion ratio.

        A_throat = T_sl / (Pc * CF)
        A_exit   = A_throat * ER

    Args:
        Tsl (float): sea-level thrust  [N]
        Pc  (float): chamber pressure  [Pa]
        CF  (float): thrust coefficient  [-]
        ER  (float): nozzle expansion ratio  A_exit / A_throat  [-]

    Returns:
        tuple: (A_throat [m2], A_exit [m2])
    """
    A_throat = Tsl / (Pc * CF)
    A_exit   = A_throat * ER
    return A_throat, A_exit


_MACH_TABLE = np.array([
    0.0834, 0.4964, 0.8750, 0.9438, 0.9954, 1.0297, 1.0813, 1.1329,
    1.3738, 1.6492, 1.8557, 2.0795, 2.3204, 2.5786, 2.8712, 3.1638,
    3.4736, 3.8178, 4.1792, 4.5406, 4.9193, 5.2807, 5.6765, 6.0895,
    6.5025, 6.9156, 7.3114, 7.7244, 8.1374, 8.5504, 8.9635, 9.3937,
    9.8067,10.2025,10.7360,11.1662,11.5792,12.0095,12.4225])

_CD_TABLE = np.array([
    0.40382, 0.40385, 0.40735, 0.43164, 0.47502, 0.50972, 0.55310, 0.59648,
    0.61905, 0.58784, 0.54969, 0.51327, 0.47859, 0.44391, 0.41097, 0.38150,
    0.35376, 0.32950, 0.30697, 0.28618, 0.26886, 0.25501, 0.24116, 0.23078,
    0.22214, 0.21523, 0.21006, 0.20141, 0.19624, 0.18933, 0.18416, 0.18072,
    0.17728, 0.17211, 0.16868, 0.16698, 0.16354, 0.16184, 0.16013])

_cd_spline = CubicSpline(_MACH_TABLE, _CD_TABLE, bc_type='natural')


def drag_coefficient(mach):
    """
    Return Cd from 39-point Mach table via natural cubic spline.
    Mach is clamped to table bounds to prevent polynomial extrapolation.

    Args:
        mach (float): Mach number

    Returns:
        float: drag coefficient
    """
    return float(_cd_spline(np.clip(mach, _MACH_TABLE[0], _MACH_TABLE[-1])))


def _boost_slopes(V, alfa, y, t, Ms, tb, C2, c0, Au, F0, Aref):
    """Compute state derivatives during boost phase."""
    f    = C2
    frac = 1.0 + f * (1.0 - t / tb)
    rho  = air_density(y)
    mach = V / math.sqrt(GAMMA * R * air_temperature(y)) if V > 0 else _MACH_TABLE[0]
    Cd   = drag_coefficient(mach)
    dP   = ATM - air_pressure(y)
    c    = c0 * (1.0 + Au * dP / F0)
    ag   = (c * f / (tb * frac * g)
            - 0.5 * rho * V**2 * Aref * Cd / Ms / g / frac
            - math.sin(alfa * rad))
    dV = ag * g
    da = -math.cos(alfa * rad) * g / V if V != 0 else 0.0
    dy = V * math.sin(alfa * rad)
    dx = V * math.cos(alfa * rad)
    return dV, da, dy, dx


def _coast_slopes(V, alfa, y, Ms, Aref):
    """Compute state derivatives during coast (unpowered) phase."""
    rho  = air_density(y)
    mach = V / math.sqrt(GAMMA * R * air_temperature(y)) if V > 0 else _MACH_TABLE[0]
    Cd   = drag_coefficient(mach)
    ag   = (-0.5 * rho * V**2 * Aref * Cd / Ms / g
            - math.sin(alfa * rad))
    dV = ag * g
    da = -math.cos(alfa * rad) * g / V if V != 0 else 0.0
    dy = V * math.sin(alfa * rad)
    dx = V * math.cos(alfa * rad)
    return dV, da, dy, dx


def _heun_boost(V, alfa, y, x, t, dt, Ms, tb, C2, c0, Au, F0, Aref):
    """Single Heun step during boost phase.

    Args:
        V, alfa, y, x (float): current state (velocity, angle, altitude, range)
        t, dt         (float): current time and step size  [s]
        Ms            (float): structural mass  [kg]
        tb            (float): burn time  [s]
        C2            (float): propellant mass ratio Mp/Ms
        c0            (float): exhaust velocity  [m/s]
        Au            (float): nozzle exit area  [m2]
        F0            (float): reference sea-level thrust  [N]
        Aref          (float): airframe reference area  [m2]

    Returns:
        tuple: updated (V, alfa, y, x)
    """
    dV1, da1, dy1, dx1 = _boost_slopes(V,  alfa,  y,  t, Ms, tb, C2, c0, Au, F0, Aref)
    V1 = V + dV1*dt; a1 = alfa + da1*dt
    y1 = max(y + dy1*dt, 0.0); x1 = x + dx1*dt
    dV2, da2, dy2, dx2 = _boost_slopes(V1, a1, y1, t, Ms, tb, C2, c0, Au, F0, Aref)
    return (V    + 0.5*(dV1+dV2)*dt,
            alfa + 0.5*(da1+da2)*dt,
            y    + 0.5*(dy1 + V1*math.sin(a1*rad))*dt,
            x    + 0.5*(dx1 + V1*math.cos(a1*rad))*dt)


def _heun_coast(V, alfa, y, x, dt, Ms, Aref):
    """Single Heun step during coast phase.

    Args:
        V, alfa, y, x (float): current state
        dt            (float): step size  [s]
        Ms            (float): dry mass  [kg]
        Aref          (float): airframe reference area  [m2]

    Returns:
        tuple: updated (V, alfa, y, x)
    """
    dV1, da1, dy1, dx1 = _coast_slopes(V,  alfa,  y,  Ms, Aref)
    V1 = V + dV1*dt; a1 = alfa + da1*dt
    y1 = y + dy1*dt;  x1 = x + dx1*dt
    dV2, da2, dy2, dx2 = _coast_slopes(V1, a1, y1, Ms, Aref)
    return (V    + 0.5*(dV1+dV2)*dt,
            alfa + 0.5*(da1+da2)*dt,
            y    + 0.5*(dy1 + V1*math.sin(a1*rad))*dt,
            x    + 0.5*(dx1 + V1*math.cos(a1*rad))*dt)


def simulate_apogee(Tsl, tb, epsilon, c0, Pc, CF, ER, alfa0, d_cm, dt=0.1):
    """Fast apogee estimate (dt=0.1 s) for feasibility curve generation.

    Nozzle exit area is derived from thrust, chamber pressure and expansion
    ratio so that the geometry scales consistently with motor size.

    Args:
        Tsl     (float): sea-level thrust  [N]
        tb      (float): burn time  [s]
        epsilon (float): structural fraction Ms/M0
        c0      (float): effective exhaust velocity  [m/s]
        Pc      (float): chamber pressure  [Pa]
        CF      (float): thrust coefficient
        ER      (float): nozzle expansion ratio  A_exit/A_throat
        alfa0   (float): launch angle  [deg]
        d_cm    (float): airframe diameter  [cm]
        dt      (float): integration step  [s]

    Returns:
        float: apogee altitude  [m]
    """
    Mp   = Tsl * tb / c0
    Ms   = epsilon * Mp / (1.0 - epsilon)
    C2   = (1.0 - epsilon) / epsilon
    F0   = C2 * Ms * c0 / tb
    Aref = PI / 4.0 * (d_cm / 100.0)**2
    _, Au = nozzle_exit_area(Tsl, Pc, CF, ER)

    t = 0.0; V = 0.0; alfa = alfa0; y = 0.0; x = 0.0
    while t <= tb:
        V, alfa, y, x = _heun_boost(V, alfa, y, x, t, dt, Ms, tb, C2, c0, Au, F0, Aref)
        t += dt
    while V >= 0:
        V, alfa, y, x = _heun_coast(V, alfa, y, x, dt, Ms, Aref)
        t += dt
    return y


def simulate_trajectory(Tsl, tb, epsilon, c0, Pc, CF, ER, alfa0, d_cm, dt=0.01):
    """Full high-fidelity trajectory at dt=0.01 s.
    Saves one row every 50 steps (0.5 s output resolution).

    Nozzle exit area scales with thrust via Pc, CF, and ER.

    Args:
        Tsl     (float): sea-level thrust  [N]
        tb      (float): burn time  [s]
        epsilon (float): structural fraction Ms/M0
        c0      (float): effective exhaust velocity  [m/s]
        Pc      (float): chamber pressure  [Pa]
        CF      (float): thrust coefficient
        ER      (float): nozzle expansion ratio  A_exit/A_throat
        alfa0   (float): launch angle  [deg]
        d_cm    (float): airframe diameter  [cm]
        dt      (float): integration step  [s]

    Returns:
        dict with keys:
            t, h, V, Mach, Drag, alfa, Cd, x  - trajectory arrays (every 0.5 s)
            y_bo, V_bo, alfa_bo            - burnout scalars
            y_apo, x_apo                   - apogee scalars  [m]
            Ms, Mp, M0, C2, TW             - mass and performance scalars
            A_throat, Au                   - nozzle geometry  [m2]
    """
    Mp   = Tsl * tb / c0
    Ms   = epsilon * Mp / (1.0 - epsilon)
    C2   = (1.0 - epsilon) / epsilon
    M0   = Mp + Ms
    TW   = Tsl / (M0 * g)
    F0   = C2 * Ms * c0 / tb
    Aref = PI / 4.0 * (d_cm / 100.0)**2
    A_throat, Au = nozzle_exit_area(Tsl, Pc, CF, ER)

    t = 0.0; V = 0.0; alfa = alfa0; y = 0.0; x = 0.0; step = 0
    ts=[]; hs=[]; Vs=[]; Machs=[]; Drags=[]; alfas=[]; Cds=[]; xs=[]

    def record():
        rho  = air_density(y)
        mach = V / math.sqrt(GAMMA * R * air_temperature(y)) if V > 0 else _MACH_TABLE[0]
        Cd   = drag_coefficient(mach)
        ts.append(t); hs.append(y); Vs.append(V); Machs.append(mach)
        Drags.append(0.5 * rho * Cd * Aref * V**2); alfas.append(alfa); Cds.append(Cd); xs.append(x)

    while t <= tb:
        if step % 50 == 0: record()
        V, alfa, y, x = _heun_boost(V, alfa, y, x, t, dt, Ms, tb, C2, c0, Au, F0, Aref)
        t += dt; step += 1

    y_bo = y; V_bo = V; alfa_bo = alfa

    while V >= 0:
        if step % 50 == 0: record()
        V, alfa, y, x = _heun_coast(V, alfa, y, x, dt, Ms, Aref)
        t += dt; step += 1

    return dict(
        t=np.array(ts), h=np.array(hs), V=np.array(Vs),
        Mach=np.array(Machs), Drag=np.array(Drags),
        alfa=np.array(alfas), Cd=np.array(Cds), x=np.array(xs),
        y_bo=y_bo, V_bo=V_bo, alfa_bo=alfa_bo,
        y_apo=y, x_apo=x,
        Ms=Ms, Mp=Mp, M0=M0, C2=C2, TW=TW,
        A_throat=A_throat, Au=Au
    )


print("Physics models loaded.")
print(f"  Nozzle check (Tsl=1800 N, Pc=30bar, CF=1.5, ER=4):")
At, Ae = nozzle_exit_area(1800, 30e5, 1.5, 4.0)
print(f"    A_throat = {At*1e4:.4f} cm2   A_exit = {Ae*1e4:.4f} cm2   d_exit = {(Ae/PI*4)**0.5*100:.2f} cm")


## Feasibility curve - T$_{sl}$ x $t_b$

In [ ]:
H_target = H_target_km * 1000.0
TB_SCAN  = np.linspace(0.5, 60, 60)

curve_Tsl=[]; curve_tb=[]; curve_TW=[]; curve_Mp=[]; curve_Ms=[]
curve_At=[]; curve_Au=[]

print(f"Computing feasibility curve  |  target: {H_target_km} km")
print(f"epsilon={epsilon:.3f}  Pc={Pc/1e5:.1f} bar  CF={CF:.2f}  ER={ER:.1f}")
print()
print(f"  {'T_sl [kN]':>10}  {'tb [s]':>7}  {'Mp [kg]':>8}  {'Ms [kg]':>8}  "
      f"{'M0 [kg]':>8}  {'T/W':>6}  {'At [cm2]':>9}  {'Au [cm2]':>9}")
print("  " + "-"*78)

for Tsl in np.linspace(Tsl_min_kN, Tsl_max_kN, n_points) * 1000:
    tb_tw1 = c0 * (1.0 - epsilon) / g * 0.99
    if tb_tw1 <= 1.0: continue

    tb_lo = tb_hi = apo_prev = None
    for tb in TB_SCAN:
        if tb >= tb_tw1: break
        try:
            apo = simulate_apogee(Tsl, tb, epsilon, c0, Pc, CF, ER, alfa0, d_cm)
        except Exception:
            break
        if apo_prev is not None and (apo_prev - H_target) * (apo - H_target) < 0:
            idx = list(TB_SCAN).index(tb)
            tb_lo = TB_SCAN[idx - 1]; tb_hi = tb
            break
        apo_prev = apo

    if tb_lo is None: continue

    try:
        tb_sol = brentq(
            lambda tb: simulate_apogee(Tsl, tb, epsilon, c0, Pc, CF, ER, alfa0, d_cm) - H_target,
            tb_lo, tb_hi, xtol=0.05)
        Mp_s = Tsl * tb_sol / c0
        Ms_s = epsilon * Mp_s / (1.0 - epsilon)
        M0_s = Mp_s + Ms_s
        TW_s = Tsl / (M0_s * g)
        At_s, Au_s = nozzle_exit_area(Tsl, Pc, CF, ER)
        curve_Tsl.append(Tsl/1000); curve_tb.append(tb_sol)
        curve_TW.append(TW_s); curve_Mp.append(Mp_s); curve_Ms.append(Ms_s)
        curve_At.append(At_s*1e4); curve_Au.append(Au_s*1e4)
        print(f"  {Tsl/1000:>10.3f}  {tb_sol:>7.2f}  {Mp_s:>8.3f}  {Ms_s:>8.3f}  "
              f"{M0_s:>8.3f}  {TW_s:>6.2f}  {At_s*1e4:>9.4f}  {Au_s*1e4:>9.4f}")
    except Exception:
        pass

print(f"\n{len(curve_Tsl)} points on feasibility curve.")

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle(
    f"Feasibility curve  |  Target: {H_target_km:.0f} km  "
    f"  epsilon={epsilon:.2f}  d={d_cm:.0f} cm  alpha0={alfa0:.0f} deg  "
    f"  Pc={Pc/1e5:.0f} bar  CF={CF:.2f}  ER={ER:.1f}",
    fontsize=11, fontweight='bold')

axes[0,0].plot(curve_tb, curve_Tsl, 'b-o', lw=2, ms=5)
axes[0,0].set_xlabel('Burn time  tb [s]'); axes[0,0].set_ylabel('T_sl [kN]')
axes[0,0].set_title('Thrust x Burn time'); axes[0,0].grid(True, alpha=0.3)

axes[0,1].plot(curve_Tsl, curve_TW, 'r-o', lw=2, ms=5)
axes[0,1].axhline(1.5, color='gray', ls='--', lw=1, label='T/W = 1.5')
axes[0,1].set_xlabel('T_sl [kN]'); axes[0,1].set_ylabel('T/W')
axes[0,1].set_title('Thrust-to-weight ratio')
axes[0,1].legend(fontsize=9); axes[0,1].grid(True, alpha=0.3)

axes[0,2].plot(curve_Tsl, curve_Mp, 'g-o', lw=2, ms=5, label='Propellant  Mp')
axes[0,2].plot(curve_Tsl, curve_Ms, 'b--o', lw=1.5, ms=4, label='Structure  Ms')
axes[0,2].set_xlabel('T_sl [kN]'); axes[0,2].set_ylabel('Mass [kg]')
axes[0,2].set_title('Propellant and structural mass')
axes[0,2].legend(fontsize=9); axes[0,2].grid(True, alpha=0.3)

axes[1,0].plot(curve_Tsl, curve_At, 'b-o', lw=2, ms=5, label='A_throat')
axes[1,0].plot(curve_Tsl, curve_Au, 'r--o', lw=1.5, ms=4, label='A_exit')
axes[1,0].set_xlabel('T_sl [kN]'); axes[1,0].set_ylabel('Area [cm2]')
axes[1,0].set_title('Nozzle areas vs thrust')
axes[1,0].legend(fontsize=9); axes[1,0].grid(True, alpha=0.3)

axes[1,1].plot(curve_Tsl, [np.sqrt(a/np.pi)*2*10 for a in curve_Au], 'r-o', lw=2, ms=5)
axes[1,1].set_xlabel('T_sl [kN]'); axes[1,1].set_ylabel('d_exit [cm]')
axes[1,1].set_title('Nozzle exit diameter vs thrust'); axes[1,1].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()


## Select design point
Look at the feasibility curve above and set the desired burn time.

In [ ]:
tb_selected = 27

## Full trajectory

In [ ]:
_idx = np.argsort(curve_tb)
Tsl_selected = float(np.interp(
    tb_selected,
    np.array(curve_tb)[_idx],
    np.array(curve_Tsl)[_idx])) * 1000.0

res = simulate_trajectory(Tsl_selected, tb_selected, epsilon, c0, Pc, CF, ER, alfa0, d_cm)

print("=" * 56)
print("  Design point")
print("=" * 56)
print(f"  Thrust    T_sl  =  {Tsl_selected/1000:.4f} kN")
print(f"  Burn time tb    =  {tb_selected:.1f} s")
print(f"  Propellant Mp   =  {res['Mp']:.3f} kg")
print(f"  Structure  Ms   =  {res['Ms']:.3f} kg")
print(f"  Lift-off   M0   =  {res['M0']:.3f} kg")
print(f"  C2 (Mp/Ms)      =  {res['C2']:.4f}")
print(f"  T/W             =  {res['TW']:.3f}")
print()
print(f"  A_throat        =  {res['A_throat']*1e4:.4f} cm2  "
      f"  d_throat = {(res['A_throat']/PI*4)**0.5*100:.3f} cm")
print(f"  A_exit          =  {res['Au']*1e4:.4f} cm2  "
      f"  d_exit   = {(res['Au']/PI*4)**0.5*100:.3f} cm")
print()
print(f"  Burnout   h     =  {res['y_bo']/1000:.3f} km")
print(f"            V     =  {res['V_bo']:.2f} m/s")
print(f"            alfa  =  {res['alfa_bo']:.4f} deg")
print()
print(f"  Apogee    h     =  {res['y_apo']/1000:.4f} km  (target: {H_target_km} km)")
print("=" * 56)
print()

print(f"{'t [s]':>7}  {'h [m]':>10}  {'V [m/s]':>9}  {'Mach':>7}  "
      f"{'Drag [N]':>9}  {'alfa [deg]':>10}  {'Cd':>8}")
print("-" * 72)
for i in range(len(res['t'])):
    print(f"  {res['t'][i]:>5.1f}  {res['h'][i]:>10.2f}  {res['V'][i]:>9.3f}  "
          f"{res['Mach'][i]:>7.4f}  {res['Drag'][i]:>9.2f}  "
          f"{res['alfa'][i]:>10.4f}  {res['Cd'][i]:>8.5f}")

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle(
    f"Trajectory  |  T_sl={Tsl_selected/1000:.3f} kN  tb={tb_selected} s  "
    f"Ms={res['Ms']:.1f} kg  Mp={res['Mp']:.1f} kg  "
    f"Apogee={res['y_apo']/1000:.2f} km  "
    f"d_exit={( res['Au']/PI*4)**0.5*100:.2f} cm",
    fontsize=10, fontweight='bold')

t = res['t']

axes[0,0].plot(t, res['h']/1000, 'b-', lw=1.5)
axes[0,0].axvline(tb_selected, color='r', ls='--', lw=1, label=f'Burnout  t={tb_selected} s')
axes[0,0].axhline(res['y_apo']/1000, color='g', ls=':', lw=1,
                  label=f"Apogee  {res['y_apo']/1000:.2f} km")
axes[0,0].set_xlabel('t [s]'); axes[0,0].set_ylabel('h [km]')
axes[0,0].set_title('Altitude'); axes[0,0].legend(fontsize=8); axes[0,0].grid(True, alpha=0.3)

axes[0,1].plot(t, res['V'], 'r-', lw=1.5)
axes[0,1].axvline(tb_selected, color='r', ls='--', lw=1)
axes[0,1].set_xlabel('t [s]'); axes[0,1].set_ylabel('V [m/s]')
axes[0,1].set_title('Velocity'); axes[0,1].grid(True, alpha=0.3)

axes[0,2].plot(t, res['Mach'], 'g-', lw=1.5)
axes[0,2].axvline(tb_selected, color='r', ls='--', lw=1)
axes[0,2].axhline(1.0, color='k', ls=':', lw=0.8, label='Mach 1')
axes[0,2].set_xlabel('t [s]'); axes[0,2].set_ylabel('Mach')
axes[0,2].set_title('Mach number'); axes[0,2].legend(fontsize=8); axes[0,2].grid(True, alpha=0.3)

x_axis = res['x']/1000
axes[1,0].plot(x_axis, res['h']/1000, 'b-', lw=1.5)
axes[1,0].scatter([res['x_apo']/1000], [res['y_apo']/1000],
                  color='g', s=80, marker='*', zorder=5, label='Apogee')
axes[1,0].set_xlabel('x [km]'); axes[1,0].set_ylabel('h [km]')
axes[1,0].set_title('x-h trajectory'); axes[1,0].legend(fontsize=8); axes[1,0].grid(True, alpha=0.3)

axes[1,1].plot(t, res['Drag']/1000, 'm-', lw=1.5)
axes[1,1].axvline(tb_selected, color='r', ls='--', lw=1)
axes[1,1].set_xlabel('t [s]'); axes[1,1].set_ylabel('Drag [kN]')
axes[1,1].set_title('Aerodynamic drag'); axes[1,1].grid(True, alpha=0.3)

axes[1,2].plot(t, res['alfa'], 'c-', lw=1.5)
axes[1,2].axvline(tb_selected, color='r', ls='--', lw=1)
axes[1,2].set_xlabel('t [s]'); axes[1,2].set_ylabel('Flight path angle [deg]')
axes[1,2].set_title('Flight path angle'); axes[1,2].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()
